# Modelo 1 — Bayesian MCMC (PyMC): Francia vs Paraguay

Predicción del resultado (local / empate / visitante) de partidos de fútbol internacional
usando un modelo de regresión logística multinomial **Bayesiano** estimado con **MCMC (NUTS)** en PyMC.

Dataset: [martj42/international_results](https://github.com/martj42/international_results)

**Codificación del target:**
- `0` = gana `home_team`
- `1` = empate
- `2` = gana `away_team`

> Nota: este notebook está pensado para **Google Colab**. Todo el código de las secciones
> se calcula al ejecutarse; no hay resultados "hardcodeados".


## 1. Instalación e importación de librerías

Instalamos `pymc`, `arviz` y `pytensor` (Colab no los trae por defecto) y dejamos fijo
`RANDOM_SEED = 42` para que los resultados sean reproducibles.

In [ ]:
# Instalación de librerías (Google Colab no trae PyMC/ArviZ por defecto)
!pip install -q "pymc>=5.10" arviz pytensor scikit-learn

import warnings
warnings.filterwarnings("ignore")

import os
import subprocess
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import pymc as pm
import arviz as az
import pytensor.tensor as pt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("PyMC version:", pm.__version__)
print("ArviZ version:", az.__version__)


## 2. Clonar el repositorio de GitHub y cargar el dataset

`REPO_URL` apunta al repo de datos históricos. `find_results_csv` busca de forma
flexible el archivo `results.csv` (o equivalente) dentro del repo clonado: primero por
nombre típico, y si no lo encuentra, por las columnas que contiene.

In [ ]:
# ============================================================
# 2. CLONAR REPOSITORIO Y CARGAR EL DATASET
# ============================================================
# Pega aquí el link del repo de GitHub que contiene los datos históricos.
REPO_URL = "https://github.com/martj42/international_results"

REPO_DIR = "/content/international_results_repo"

if not os.path.exists(REPO_DIR):
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    except Exception as e:
        print("No se pudo clonar el repo automáticamente:", e)
else:
    print("El repositorio ya existe localmente, se omite el clonado.")


def find_results_csv(search_dirs):
    """
    Busca de forma flexible un archivo CSV de resultados dentro de una lista de
    directorios. Prioriza nombres típicos ('results.csv') y si no los encuentra,
    busca cualquier CSV que contenga las columnas esperadas (home_team, away_team, ...).
    """
    candidate_names = ["results.csv", "result.csv", "matches.csv", "international_results.csv"]
    csv_files = []
    for base_dir in search_dirs:
        if not os.path.isdir(base_dir):
            continue
        for root, _, files in os.walk(base_dir):
            for f in files:
                if f.lower().endswith(".csv"):
                    csv_files.append(os.path.join(root, f))

    # 1. Buscar coincidencia exacta por nombre
    for name in candidate_names:
        for path in csv_files:
            if os.path.basename(path).lower() == name:
                return path

    # 2. Si no hay coincidencia por nombre, buscar por columnas esperadas
    expected_cols = {"home_team", "away_team", "home_score", "away_score"}
    for path in csv_files:
        try:
            sample = pd.read_csv(path, nrows=5)
            if expected_cols.issubset(set(c.lower() for c in sample.columns)):
                return path
        except Exception:
            continue

    raise FileNotFoundError(
        "No se encontro un CSV con las columnas esperadas. "
        "Revisa REPO_URL o coloca manualmente la ruta del archivo en csv_path."
    )


# Busca primero en el repo clonado y, como respaldo, en el directorio actual
csv_path = find_results_csv([REPO_DIR, "."])
print("Archivo de resultados encontrado en:", csv_path)

raw_df = pd.read_csv(csv_path)
raw_df.columns = [c.strip().lower() for c in raw_df.columns]
print("Shape original:", raw_df.shape)
raw_df.head()


## 3. Limpieza del dataset

- Normalizamos nombres de columnas.
- Convertimos `date` a datetime y ordenamos cronológicamente (paso crítico: todo el
  feature engineering posterior depende de este orden).
- Eliminamos partidos sin marcador.
- Creamos la variable objetivo `target` (0 = local, 1 = empate, 2 = visitante).
- Columnas opcionales (`tournament`, `neutral`) se manejan de forma flexible por si
  no existieran en el dataset.

In [ ]:
# ============================================================
# 3. LIMPIEZA DEL DATASET
# ============================================================
df = raw_df.copy()

# Normalizar nombres de columnas por si vienen con variantes
rename_map = {
    "hometeam": "home_team", "awayteam": "away_team",
    "homescore": "home_score", "awayscore": "away_score",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

required_cols = ["date", "home_team", "away_team", "home_score", "away_score"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas obligatorias en el dataset: {missing}")

# Convertir fecha y descartar filas sin fecha valida
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])

# Ordenar cronologicamente (necesario para que el feature engineering no tenga leakage)
df = df.sort_values("date").reset_index(drop=True)

# Eliminar partidos sin marcador valido
df = df.dropna(subset=["home_score", "away_score"])
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)

# Columnas opcionales manejadas de forma flexible
if "tournament" not in df.columns:
    df["tournament"] = "Unknown"

if "neutral" not in df.columns:
    df["neutral"] = False
else:
    df["neutral"] = (
        df["neutral"].astype(str).str.upper().map({"TRUE": True, "FALSE": False}).fillna(False)
    )
df["neutral"] = df["neutral"].astype(int)

# id unico de partido (se usa para reconstruir las features mas adelante)
df["match_id"] = np.arange(len(df))

# Variable objetivo (target)
# 0 = gana home_team | 1 = empate | 2 = gana away_team
conditions = [
    df["home_score"] > df["away_score"],
    df["home_score"] == df["away_score"],
    df["home_score"] < df["away_score"],
]
df["target"] = np.select(conditions, [0, 1, 2]).astype(int)

print(f"Dataset limpio: {df.shape[0]} partidos, {df.shape[1]} columnas")
print(df["target"].value_counts(normalize=True).rename("proporcion"))
df[["date", "home_team", "away_team", "home_score", "away_score", "target"]].head()


## 4. Feature engineering (sin data leakage)

**Estrategia:**
1. Convertimos cada partido en 2 filas (formato "largo"): una desde la perspectiva del
   equipo local y otra desde la del visitante.
2. Para cada equipo, calculamos métricas de forma con ventanas móviles de **3, 5, 10, 15
   y 20 partidos anteriores**. Usamos `shift(1)` **antes** del `rolling`, de modo que el
   partido actual nunca se usa para calcular sus propias variables (evita data leakage).
3. Reconstruimos el formato ancho (una fila por partido) uniendo las métricas del
   local (`home_*`), del visitante (`away_*`) y sus diferencias (`diff_*`).

Con 12 métricas × 5 ventanas × 3 perspectivas (local / visitante / diferencia)
obtenemos **180 variables**, muy por encima de las 100 solicitadas.

In [ ]:
# ============================================================
# 4. FEATURE ENGINEERING (SIN DATA LEAKAGE)
# ============================================================
WINDOWS = [3, 5, 10, 15, 20]

# --- 4.1 Formato largo: una fila por equipo y partido ---
long_cols = ["match_id", "date", "home_team", "away_team", "home_score", "away_score", "tournament", "neutral"]

home_long = df[long_cols].rename(columns={
    "home_team": "team", "away_team": "opponent",
    "home_score": "goals_for", "away_score": "goals_against",
})
home_long["is_home"] = 1

away_long = df[long_cols].rename(columns={
    "away_team": "team", "home_team": "opponent",
    "away_score": "goals_for", "home_score": "goals_against",
})
away_long["is_home"] = 0

long_df = pd.concat([home_long, away_long], ignore_index=True)
long_df = long_df.sort_values(["team", "date", "match_id"]).reset_index(drop=True)

# --- 4.2 Metricas base por partido (antes de aplicar ventanas) ---
long_df["goal_diff"] = long_df["goals_for"] - long_df["goals_against"]
long_df["win"] = (long_df["goals_for"] > long_df["goals_against"]).astype(int)
long_df["draw"] = (long_df["goals_for"] == long_df["goals_against"]).astype(int)
long_df["loss"] = (long_df["goals_for"] < long_df["goals_against"]).astype(int)
long_df["points"] = long_df["win"] * 3 + long_df["draw"] * 1
long_df["clean_sheet"] = (long_df["goals_against"] == 0).astype(int)
long_df["failed_to_score"] = (long_df["goals_for"] == 0).astype(int)

# 9 metricas "mean/rate" + 3 metricas de desviacion estandar = 12 metricas por ventana
BASE_METRICS = ["goals_for", "goals_against", "goal_diff", "points",
                "win", "draw", "loss", "clean_sheet", "failed_to_score"]
STD_METRICS = ["goals_for", "goals_against", "goal_diff"]

# --- 4.3 Rolling con shift(1): usa SOLO partidos anteriores al actual ---
feature_cols_long = []
grouped = long_df.groupby("team", group_keys=False)

t0 = time.time()
new_cols = {}
for w in WINDOWS:
    for col in BASE_METRICS:
        feat_name = f"{col}_mean_{w}"
        new_cols[feat_name] = grouped[col].apply(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        feature_cols_long.append(feat_name)
    for col in STD_METRICS:
        feat_name = f"{col}_std_{w}"
        new_cols[feat_name] = grouped[col].apply(lambda s: s.shift(1).rolling(w, min_periods=1).std())
        feature_cols_long.append(feat_name)

long_df = pd.concat([long_df, pd.DataFrame(new_cols, index=long_df.index)], axis=1)
print(f"Variables de forma por equipo calculadas en {time.time() - t0:.1f}s -> {len(feature_cols_long)} columnas")

# --- 4.4 Volver a formato ancho: features del local, del visitante y diferencias ---
rename_home = {c: f"home_{c}" for c in feature_cols_long}
rename_away = {c: f"away_{c}" for c in feature_cols_long}

home_features = long_df.loc[long_df.is_home == 1, ["match_id"] + feature_cols_long].rename(columns=rename_home)
away_features = long_df.loc[long_df.is_home == 0, ["match_id"] + feature_cols_long].rename(columns=rename_away)

df = df.merge(home_features, on="match_id", how="left").merge(away_features, on="match_id", how="left")

diff_data = {f"diff_{c}": df[f"home_{c}"] - df[f"away_{c}"] for c in feature_cols_long}
df = pd.concat([df, pd.DataFrame(diff_data, index=df.index)], axis=1)
diff_cols = list(diff_data.keys())

all_feature_cols = [f"home_{c}" for c in feature_cols_long] + [f"away_{c}" for c in feature_cols_long] + diff_cols
print(f"Total de variables generadas: {len(all_feature_cols)}")

# Partidos al inicio del historico de un equipo no tienen ventana previa -> NaN.
# Se rellenan con 0 (equivalente a "sin informacion previa disponible").
df[all_feature_cols] = df[all_feature_cols].fillna(0)

assert len(all_feature_cols) > 100, "Se esperaban mas de 100 variables"
df[all_feature_cols].describe().T.head()


## 5. Separación temporal train / test (80% / 20%)

Nada de `train_test_split` aleatorio: usamos los primeros 80% de partidos
(cronológicamente) para entrenar y el último 20% para test, simulando un escenario
real de predicción "hacia adelante en el tiempo".

In [ ]:
# ============================================================
# 5. SEPARACION TEMPORAL TRAIN / TEST (80% / 20%)
# ============================================================
df_model = df.sort_values("date").reset_index(drop=True)

split_idx = int(len(df_model) * 0.8)

train_df = df_model.iloc[:split_idx].copy()
test_df = df_model.iloc[split_idx:].copy()

print(f"Train: {train_df.shape[0]} partidos "
      f"({train_df['date'].min().date()} a {train_df['date'].max().date()})")
print(f"Test:  {test_df.shape[0]} partidos "
      f"({test_df['date'].min().date()} a {test_df['date'].max().date()})")

y_train = train_df["target"].values.astype(int)
y_test = test_df["target"].values.astype(int)


## 6. Preparación para PyMC: selección de variables + escalado

Con 180 variables, un MCMC completo sería muy lento (NUTS tiene que explorar un
espacio de parámetros de alta dimensión, y muchas de estas variables están
correlacionadas entre sí — p.ej. la media de goles en ventanas de 15 y 20 partidos).

**Estrategia elegida:** reducir a las `TOP_K` variables más informativas según la
importancia de un `RandomForestClassifier` entrenado **solo con el train set** (así no
hay fuga de información desde el test). Esto:

- Acelera drásticamente el muestreo MCMC (menos parámetros → geometría del posterior
  más simple → NUTS converge más rápido).
- Reduce colinealidad entre variables redundantes.
- Actúa como un paso de selección de variables antes de imponer los priors Bayesianos.

*Alternativa (no implementada por defecto):* usar priors tipo "horseshoe" sobre las
180 variables para regularización automática sin selección manual. Es más elegante
pero mucho más costoso computacionalmente con NUTS — razonable solo con más tiempo de
cómputo disponible.

In [ ]:
# ============================================================
# 6. SELECCION DE VARIABLES + ESCALADO PARA PyMC
# ============================================================
TOP_K = 40

X_train_full = train_df[all_feature_cols].values
X_test_full = test_df[all_feature_cols].values

rf_selector = RandomForestClassifier(
    n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1
)
rf_selector.fit(X_train_full, y_train)

importances = pd.Series(rf_selector.feature_importances_, index=all_feature_cols)
top_features = importances.sort_values(ascending=False).head(TOP_K).index.tolist()

print(f"Top {TOP_K} variables seleccionadas por importancia (Random Forest, solo train):")
for f in top_features[:15]:
    print(f"  {f}: {importances[f]:.4f}")

X_train_top = train_df[top_features].values
X_test_top = test_df[top_features].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_top)
X_test_scaled = scaler.transform(X_test_top)

print("\nShape final para PyMC -> train:", X_train_scaled.shape, "| test:", X_test_scaled.shape)


## 7. Modelo Bayesiano multinomial (regresión logística MCMC)

Regresión logística multinomial con clase base `home_team gana` (`eta_home = 0`).
Se modelan explícitamente los otros dos "logits":

- `eta_draw = intercept_draw + X · beta_draw`
- `eta_away = intercept_away + X · beta_away`

Las probabilidades finales salen de `softmax([eta_home, eta_draw, eta_away])`.
Usamos `pm.Data` (mutable) para `X` e `y`, de modo que más adelante podamos
reutilizar el mismo modelo entrenado para predecir sobre el test set y sobre el
partido Francia–Paraguay simplemente con `pm.set_data`.

In [ ]:
# ============================================================
# 7. MODELO BAYESIANO MULTINOMIAL EN PyMC (MCMC / NUTS)
# ============================================================
coords = {"feature": top_features}

with pm.Model(coords=coords) as bayes_model:
    X_data = pm.Data("X_data", X_train_scaled)   # (n_obs, n_features), mutable
    y_data = pm.Data("y_data", y_train)          # (n_obs,), mutable

    # Priors normales para interceptos y coeficientes (clase base = gana local)
    intercept_draw = pm.Normal("intercept_draw", mu=0, sigma=2.5)
    intercept_away = pm.Normal("intercept_away", mu=0, sigma=2.5)

    beta_draw = pm.Normal("beta_draw", mu=0, sigma=1, dims="feature")
    beta_away = pm.Normal("beta_away", mu=0, sigma=1, dims="feature")

    eta_draw = intercept_draw + pm.math.dot(X_data, beta_draw)
    eta_away = intercept_away + pm.math.dot(X_data, beta_away)
    eta_home = pt.zeros_like(eta_draw)  # clase base fija en 0

    eta = pt.stack([eta_home, eta_draw, eta_away], axis=1)   # (n_obs, 3)
    p = pm.Deterministic("p", pm.math.softmax(eta, axis=1))  # probabilidades por clase

    y_obs = pm.Categorical("y_obs", p=p, observed=y_data)

    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=2,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
        return_inferencedata=True,
    )


## 8. Diagnóstico del modelo (MCMC)

- **`r_hat`**: debe ser muy cercano a 1.0 (idealmente ≤ 1.01). Valores más altos
  indican que las cadenas no convergieron al mismo posterior.
- **`ess_bulk` / `ess_tail`** (tamaño de muestra efectivo): cuanto más alto, más
  confiables son las estimaciones de la posterior (idealmente > 400).

In [ ]:
# ============================================================
# 8. DIAGNOSTICO DEL MODELO
# ============================================================
summary = az.summary(trace, var_names=["intercept_draw", "intercept_away", "beta_draw", "beta_away"])
print(summary)

high_rhat = summary[summary["r_hat"] > 1.01]
if len(high_rhat) > 0:
    print("\nAtencion: variables con r_hat > 1.01 (posible falta de convergencia):")
    print(high_rhat)
else:
    print("\nTodas las variables tienen r_hat <= 1.01: buena senal de convergencia.")

low_ess = summary[summary["ess_bulk"] < 400]
if len(low_ess) > 0:
    print(f"\n{len(low_ess)} variables con ess_bulk < 400 (menos precision en la estimacion):")
    print(low_ess[["ess_bulk", "ess_tail"]])


In [ ]:
# Trace plots de los interceptos (mezcla de cadenas + distribucion posterior)
az.plot_trace(trace, var_names=["intercept_draw", "intercept_away"])
plt.tight_layout()
plt.show()


In [ ]:
# Forest plots de los coeficientes beta (uno por variable seleccionada)
az.plot_forest(trace, var_names=["beta_draw"], combined=True, figsize=(8, 10))
plt.title("Coeficientes beta_draw (empate vs. gana local)")
plt.tight_layout()
plt.show()

az.plot_forest(trace, var_names=["beta_away"], combined=True, figsize=(8, 10))
plt.title("Coeficientes beta_away (gana visitante vs. gana local)")
plt.tight_layout()
plt.show()


## 9. Evaluación en el conjunto de test

Reutilizamos el modelo ya entrenado: cambiamos los datos (`pm.set_data`) por los del
test set y generamos muestras predictivas posteriores (`sample_posterior_predictive`).
La probabilidad predicha para cada partido es el promedio de `p` sobre todas las
muestras del posterior (cadenas × draws).

In [ ]:
# ============================================================
# 9. EVALUACION EN EL CONJUNTO DE TEST
# ============================================================
with bayes_model:
    pm.set_data({"X_data": X_test_scaled, "y_data": np.zeros(len(y_test), dtype=int)})
    ppc_test = pm.sample_posterior_predictive(
        trace, var_names=["p", "y_obs"], random_seed=RANDOM_SEED
    )

# Probabilidad promedio del posterior para cada partido de test: shape (n_test, 3)
p_test_mean = ppc_test.posterior_predictive["p"].mean(dim=["chain", "draw"]).values
y_pred_test = p_test_mean.argmax(axis=1)

acc = accuracy_score(y_test, y_pred_test)
print(f"Accuracy en test: {acc:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, y_pred_test,
    target_names=["Gana local", "Empate", "Gana visitante"]
))

cm = confusion_matrix(y_test, y_pred_test)
cm_df = pd.DataFrame(
    cm,
    index=["Real: local", "Real: empate", "Real: visitante"],
    columns=["Pred: local", "Pred: empate", "Pred: visitante"],
)
print("\nMatriz de confusion:")
print(cm_df)

avg_prob_per_class = p_test_mean.mean(axis=0)
print("\nProbabilidad promedio (test set) por clase:")
for cls, name in zip(range(3), ["Gana local", "Empate", "Gana visitante"]):
    print(f"  {name}: {avg_prob_per_class[cls]:.4f}")


In [ ]:
# Matriz de confusion (grafica)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["Local", "Empate", "Visitante"])
ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["Local", "Empate", "Visitante"])
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
ax.set_title("Matriz de confusion - Modelo Bayesiano MCMC")
plt.tight_layout()
plt.show()


## 10. Predicción específica: Francia vs Paraguay

`create_match_features` construye el mismo esquema de variables (`home_*`, `away_*`,
`diff_*`) que usó el entrenamiento, pero para un partido **futuro** aún no jugado:
para cada equipo tomamos sus **últimos** `w` partidos disponibles en el histórico
(incluyendo el más reciente, ya que no hay un "partido actual" que excluir).

Si un equipo no existe en el dataset, se usan valores neutros (0) en lugar de fallar.

In [ ]:
# ============================================================
# 10. PREDICCION ESPECIFICA: FRANCIA vs PARAGUAY
# ============================================================
def get_latest_team_form(team_name, long_df, windows=WINDOWS):
    """
    Calcula las metricas de forma de un equipo usando sus ULTIMOS partidos
    disponibles en el historico (incluye el partido mas reciente, ya que se
    esta proyectando un partido futuro que todavia no se jugo).
    Si el equipo no existe en el dataset, devuelve ceros (fallback flexible).
    """
    hist = long_df[long_df["team"] == team_name].sort_values("date")
    stats = {}

    if hist.empty:
        print(f"Aviso: '{team_name}' no se encontro en el historico. Se usaran valores neutros (0).")
        for w in windows:
            for col in BASE_METRICS:
                stats[f"{col}_mean_{w}"] = 0.0
            for col in STD_METRICS:
                stats[f"{col}_std_{w}"] = 0.0
        return stats

    for w in windows:
        recent = hist.tail(w)
        for col in BASE_METRICS:
            stats[f"{col}_mean_{w}"] = recent[col].mean()
        for col in STD_METRICS:
            stats[f"{col}_std_{w}"] = recent[col].std() if len(recent) > 1 else 0.0
    return stats


def create_match_features(home_team, away_team, neutral=1, all_feature_cols=all_feature_cols,
                           top_features=top_features, scaler=scaler, long_df=long_df):
    """
    Construye el vector de variables (ya reducido a top_features y escalado) para un
    partido hipotetico entre home_team y away_team, usando el mismo esquema de
    features que el modelo de entrenamiento.
    """
    home_stats = get_latest_team_form(home_team, long_df)
    away_stats = get_latest_team_form(away_team, long_df)

    row = {}
    for col, val in home_stats.items():
        row[f"home_{col}"] = val
    for col, val in away_stats.items():
        row[f"away_{col}"] = val
    for col in home_stats:
        row[f"diff_{col}"] = home_stats[col] - away_stats[col]
    # 'neutral' se deja disponible por si se agrega como feature en una version futura
    row["_neutral"] = neutral

    match_df = pd.DataFrame([row])
    for c in all_feature_cols:
        if c not in match_df.columns:
            match_df[c] = 0.0
    match_df = match_df[all_feature_cols].fillna(0.0)

    match_top = match_df[top_features].values
    match_scaled = scaler.transform(match_top)
    return match_scaled


HOME_TEAM = "France"
AWAY_TEAM = "Paraguay"

X_match_scaled = create_match_features(HOME_TEAM, AWAY_TEAM, neutral=1)

with bayes_model:
    pm.set_data({"X_data": X_match_scaled, "y_data": np.zeros(1, dtype=int)})
    ppc_match = pm.sample_posterior_predictive(trace, var_names=["p"], random_seed=RANDOM_SEED)

# (chain, draw, 1, 3) -> (n_samples, 3)
p_match_samples = ppc_match.posterior_predictive["p"].values.reshape(-1, 3)
p_match_mean = p_match_samples.mean(axis=0)

# Intervalo de credibilidad del 94% para cada clase (incertidumbre del posterior)
hdi_bounds = np.array([az.hdi(p_match_samples[:, k], hdi_prob=0.94) for k in range(3)])

results_table = pd.DataFrame({
    "resultado": [f"{HOME_TEAM} gana", "Empate", f"{AWAY_TEAM} gana"],
    "probabilidad": p_match_mean,
})
results_table["probabilidad_%"] = (results_table["probabilidad"] * 100).round(2)
results_table["hdi_94_low_%"] = (hdi_bounds[:, 0] * 100).round(2)
results_table["hdi_94_high_%"] = (hdi_bounds[:, 1] * 100).round(2)

print(f"Prediccion Bayesiana MCMC: {HOME_TEAM} vs {AWAY_TEAM}\n")
print(results_table.to_string(index=False))


In [ ]:
# Grafica de barras con las probabilidades estimadas
plt.figure(figsize=(6, 4))
colors = ["#1f77b4", "#7f7f7f", "#d62728"]
bars = plt.bar(results_table["resultado"], results_table["probabilidad_%"], color=colors)
plt.ylabel("Probabilidad (%)")
plt.title(f"Probabilidades estimadas: {HOME_TEAM} vs {AWAY_TEAM}\n(Modelo Bayesiano MCMC)")
for i, v in enumerate(results_table["probabilidad_%"]):
    plt.text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")
plt.ylim(0, max(results_table["probabilidad_%"]) + 15)
plt.tight_layout()
plt.show()


## 11. Exportar resultados

Guardamos la predicción final en `prediction_france_paraguay_bayesian.csv` con las
columnas `resultado` y `probabilidad`.

In [ ]:
# ============================================================
# 11. EXPORTAR RESULTADOS
# ============================================================
output_path = "prediction_france_paraguay_bayesian.csv"

export_df = results_table[["resultado", "probabilidad"]].copy()
export_df.to_csv(output_path, index=False)

print(f"Archivo exportado: {output_path}")
export_df
